# 06b · Attention Ablation: Early Placement

**Why this notebook exists.** The proposal compares attention placed **early, late or at all stages**. Notebook 06 tested
**Late** (after the 7x7 stage) and **Multi** (after the 28x28, 14x14 and 7x7 stages). This notebook adds the missing **Early** placement.

| Variant | What is added |
|---|---|
| **ECA-Early** | Efficient Channel Attention after the 28x28 stage only |
| **CA-Early** | Coordinate Attention after the 28x28 stage only |

Everything else is **identical to notebook 06**: same model code, identity-start gates, folds, augmentation, two-phase training,
learning-rate choice and threshold choice. The results are merged with notebook 06's five variants.

**About selection:** DFU-LiteNet's attention variant was fixed in notebook 06 by a rule set in advance, and notebook 08 was built on it.
The new variants are reported as an **extension of the ablation**. The notebook also prints which variant the same rule would pick among all six,
so any change is visible and can be discussed honestly.

**Inputs:** the dataset, **DFU 03 - Leakage-Free Split** and **DFU 06 - LiteNet Ablation**. Do not attach 05b or 07.
**Accelerator:** GPU T4 x2 · **Internet:** On · **Time:** about 45-70 minutes (10 runs).

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import confusion_matrix, roc_auc_score

# ---------------- settings (same recipe as notebook 06) ----------------
CONFIG = {
    "VARIANTS": ["ECA-Early", "CA-Early"],   # the 5 variants of notebook 06 are merged in below
    "FOLDS": [0, 1, 2, 3, 4],
    "IMG_SIZE": 224, "BATCH": 32,
    "EPOCHS_HEAD": 5,                 # phase 1: backbone frozen; attention + head train
    "EPOCHS_FT": 30,                  # phase 2: fine-tune (backbone BatchNorm stays frozen)
    "LR_HEAD": 1e-3,
    "LR_FT_OPTIONS": [1e-5, 1e-4],    # each run keeps whichever gives lower validation loss
    "PATIENCE": 7,
    "SEED": 42,
}
# -----------------------------------------------------------------------
json.dump(CONFIG, open("config.json", "w"), indent=2)   # read by train_one.py
print(json.dumps(CONFIG, indent=2))

## 1. The training script
This cell only **writes** `train_one.py`; it does not train. Each (variant, fold) run is a fresh process, so memory cannot build up.

In [ ]:
%%writefile train_one.py
# One training run: python train_one.py <variant> <test_fold>
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import json, random, sys, time, warnings
warnings.filterwarnings("ignore", message=".*shuffle=True.*")
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers, ops
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

globals().update(json.load(open("config.json")))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

PATCHES = next(Path("/kaggle/input").rglob("Patches"))
SPLITS = next(Path("/kaggle/input").rglob("splits.csv"))

# ---------------- data ----------------
splits = pd.read_csv(SPLITS)
data = splits[splits.keep].reset_index(drop=True)

def load(row):
    im = Image.open(PATCHES / row["class"] / row.file).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)

X = np.stack([load(r) for _, r in data.iterrows()]).astype(np.float32)   # raw 0-255 pixels
y = data.label.values.astype(np.float32)                                 # 1 = ulcer, 0 = normal
fold_of = data.fold.values

# ---------------- attention modules (identity at the start: gate = 2 x sigmoid(0) = 1) ----------------
class ECA(layers.Layer):
    """Efficient Channel Attention (Wang et al., CVPR 2020): which feature channels matter."""
    def __init__(self, gamma=2, b=1, **kw):
        super().__init__(**kw); self.gamma, self.b = gamma, b

    def build(self, shape):
        t = int(abs((np.log2(shape[-1]) + self.b) / self.gamma))
        k = t if t % 2 else t + 1
        self.conv = layers.Conv1D(1, k, padding="same", use_bias=False, kernel_initializer="zeros")
        self.conv.build((None, shape[-1], 1))

    def call(self, x):
        z = ops.expand_dims(ops.mean(x, axis=(1, 2)), -1)      # (batch, channels, 1)
        gate = 2.0 * ops.sigmoid(self.conv(z))                 # (batch, channels, 1)
        return x * ops.reshape(gate, (-1, 1, 1, x.shape[-1]))

class CoordAtt(layers.Layer):
    """Coordinate Attention (Hou et al., CVPR 2021): which channels matter, and where (height, width)."""
    def __init__(self, reduction=32, **kw):
        super().__init__(**kw); self.reduction = reduction

    def build(self, shape):
        c = shape[-1]; mip = max(8, c // self.reduction)
        self.conv1 = layers.Conv2D(mip, 1, use_bias=False)
        self.bn = layers.BatchNormalization(momentum=0.9)      # momentum 0.9: few steps per epoch (see notebook 05b)
        self.conv_h = layers.Conv2D(c, 1, kernel_initializer="zeros")
        self.conv_w = layers.Conv2D(c, 1, kernel_initializer="zeros")
        self.conv1.build((None, None, 1, c)); self.bn.build((None, None, 1, mip))
        self.conv_h.build((None, None, 1, mip)); self.conv_w.build((None, 1, None, mip))

    def call(self, x, training=None):
        h, w = x.shape[1], x.shape[2]
        x_h = ops.mean(x, axis=2, keepdims=True)                                     # (batch, H, 1, C)
        x_w = ops.transpose(ops.mean(x, axis=1, keepdims=True), (0, 2, 1, 3))        # (batch, W, 1, C)
        y_ = ops.relu(self.bn(self.conv1(ops.concatenate([x_h, x_w], axis=1)), training=training))
        a_h = 2.0 * ops.sigmoid(self.conv_h(y_[:, :h]))                              # (batch, H, 1, C)
        a_w = 2.0 * ops.sigmoid(self.conv_w(ops.transpose(y_[:, h:], (0, 2, 1, 3)))) # (batch, 1, W, C)
        return x * a_h * a_w

# ---------------- models ----------------
ATT = {"ECA": ECA, "CA": CoordAtt}
STAGE_ENDS = ["expanded_conv_2_add", "expanded_conv_7_add"]     # end of the 28x28 and 14x14 stages

def build_model(variant):
    base = keras.applications.MobileNetV3Small(include_top=False, weights="imagenet",
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
    kind, _, where = variant.partition("-")
    net = base
    inside = {"Multi": STAGE_ENDS, "Early": STAGE_ENDS[:1]}.get(where, [])   # Early = after the 28x28 stage only
    if inside:                                                   # insert attention inside the backbone
        def call_fn(layer, *args, **kwargs):
            out = layer(*args, **kwargs)
            return ATT[kind](name=f"att_{layer.name}")(out) if layer.name in inside else out
        net = keras.models.clone_model(base, clone_function=lambda l: l, call_function=call_fn)
        inserted = [l.name for l in net.layers if l.name.startswith("att_")]
        assert len(inserted) == len(inside), f"attention not inserted correctly: {inserted}"
    x = net.output
    if where in ("Late", "Multi"):                               # attention after the last stage
        x = ATT[kind](name="att_final")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    out = layers.Dense(1, activation="sigmoid", name="classifier")(x)
    return keras.Model(net.input, out, name=variant.replace("-", "_"))

def set_phase(model, phase):
    for l in model.layers:
        new = l.name.startswith("att_") or l.name in ("gap", "dropout", "classifier")
        if phase == "head":
            l.trainable = new                                    # phase 1: only attention + head learn
        else:
            l.trainable = not isinstance(l, layers.BatchNormalization)   # phase 2: all but backbone BatchNorm

# ---------------- one training run (same recipe as notebook 05) ----------------
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15, fill_mode="reflect"),
    layers.RandomZoom(0.1, fill_mode="reflect"),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
], name="augment")

def make_ds(ids, training):
    ds = tf.data.Dataset.from_tensor_slices((X[ids], y[ids]))
    if training:
        ds = ds.shuffle(len(ids), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH)
    if training:
        ds = ds.map(lambda a, b: (augment(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])

def choose_threshold(p, t):
    grid = np.linspace(0.01, 0.99, 99)
    bal = np.array([((p >= g)[t == 1].mean() + (p < g)[t == 0].mean()) / 2 for g in grid])
    return float(np.median(grid[bal == bal.max()]))

def run_one(variant, test_fold):
    set_seed(SEED)
    val_fold = (test_fold + 1) % 5
    tr = np.where((fold_of != test_fold) & (fold_of != val_fold))[0]
    va, te = np.where(fold_of == val_fold)[0], np.where(fold_of == test_fold)[0]
    train_ds, val_ds, test_ds = make_ds(tr, True), make_ds(va, False), make_ds(te, False)
    cw = {0: len(tr) / (2 * (y[tr] == 0).sum()), 1: len(tr) / (2 * (y[tr] == 1).sum())}
    t0 = time.time()

    model = build_model(variant)
    set_phase(model, "head")
    compile_model(model, LR_HEAD)
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=cw, verbose=0)
    head_weights, head_loss = model.get_weights(), model.evaluate(val_ds, verbose=0)[0]

    set_phase(model, "fine")
    best = {"val_loss": head_loss, "weights": head_weights, "chosen": "head-only", "lr": None, "epochs": EPOCHS_HEAD}
    lr_losses = {}
    for lr in LR_FT_OPTIONS:
        set_seed(SEED)
        model.set_weights(head_weights)
        compile_model(model, keras.optimizers.schedules.CosineDecay(lr, EPOCHS_FT * len(train_ds)))
        stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
        h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=cw, callbacks=[stop], verbose=0)
        lr_losses[lr] = min(h.history["val_loss"])
        if lr_losses[lr] < best["val_loss"]:
            best = {"val_loss": lr_losses[lr], "weights": model.get_weights(), "chosen": "fine-tuned",
                    "lr": lr, "epochs": EPOCHS_HEAD + len(h.history["loss"])}
    model.set_weights(best["weights"])
    train_min = (time.time() - t0) / 60

    thr = choose_threshold(model.predict(val_ds, verbose=0).ravel(), y[va])
    p = model.predict(test_ds, verbose=0).ravel()
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y[te], pred, labels=[0, 1]).ravel()
    row = {"model": variant, "fold": test_fold, "threshold": thr, "chosen": best["chosen"], "lr_ft": best["lr"], "epochs": best["epochs"],
           "val_loss_head": head_loss, **{f"val_loss_lr{lr:g}": v for lr, v in lr_losses.items()}, "val_loss_best": best["val_loss"],
           "train_min": round(train_min, 2), "n_test": len(te), "tp": tp, "fn": fn, "tn": tn, "fp": fp,
           "accuracy": accuracy_score(y[te], pred), "precision": precision_score(y[te], pred, zero_division=0),
           "sensitivity": recall_score(y[te], pred), "specificity": tn / (tn + fp),
           "f1": f1_score(y[te], pred), "auc": roc_auc_score(y[te], p)}
    preds = pd.DataFrame({"model": variant, "fold": test_fold, "file": data.file[te].values,
                          "class": data["class"][te].values, "label": y[te].astype(int), "prob_ulcer": p, "pred": pred})
    return row, preds, model

# ---------------- size, FLOPs and speed ----------------
def count_flops(model):
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        fn = tf.function(lambda t: model(t, training=False))
        frozen = convert_variables_to_constants_v2(fn.get_concrete_function(tf.TensorSpec([1, IMG_SIZE, IMG_SIZE, 3], tf.float32)))
        B = tf.compat.v1.profiler.ProfileOptionBuilder
        opts = B(B.float_operation()).with_empty_output().build()
        return tf.compat.v1.profiler.profile(graph=frozen.graph, run_meta=tf.compat.v1.RunMetadata(), cmd="op", options=opts).total_float_ops
    except Exception as e:
        print("   FLOPs not available:", type(e).__name__)
        return None

def latency_ms(model, device, runs=100):
    with tf.device(device):
        infer = tf.function(lambda t: model(t, training=False))
        x = tf.constant(X[:1])
        for _ in range(10): infer(x).numpy()
        t = time.perf_counter()
        for _ in range(runs): infer(x).numpy()
        return (time.perf_counter() - t) / runs * 1000

def efficiency(model, name):
    flops = count_flops(model)
    return {"model": name, "params_M": model.count_params() / 1e6, "size_mb_fp32": model.count_params() * 4 / 1e6,
            "gflops": flops / 1e9 if flops else None,
            "latency_gpu_ms": latency_ms(model, "/GPU:0") if tf.config.list_physical_devices("GPU") else None,
            "latency_cpu_ms": latency_ms(model, "/CPU:0", runs=30)}

if __name__ == "__main__":
    name, k = sys.argv[1], int(sys.argv[2])
    RES, PRED, EFF = Path("results_per_fold.csv"), Path("test_predictions_all.csv"), Path("efficiency.csv")
    row, preds, model = run_one(name, k)
    if RES.exists():
        pd.DataFrame([row]).reindex(columns=pd.read_csv(RES, nrows=0).columns).to_csv(RES, mode="a", header=False, index=False)
    else:
        pd.DataFrame([row]).to_csv(RES, index=False)
    preds.to_csv(PRED, mode="a", header=not PRED.exists(), index=False)
    os.makedirs("weights", exist_ok=True)                        # weights only (no optimizer state), reused by notebook 08
    np.savez_compressed(f"weights/{name}_fold{k}.npz", threshold=row["threshold"], **{f"w{i}": w for i, w in enumerate(model.get_weights())})
    if not (EFF.exists() and name in set(pd.read_csv(EFF).model)):
        pd.DataFrame([efficiency(model, name)]).to_csv(EFF, mode="a", header=not EFF.exists(), index=False)
    print(f"RESULT {name:10s} fold {k} | acc {row['accuracy']:.3f}  sens {row['sensitivity']:.3f}  spec {row['specificity']:.3f}"
          f"  F1 {row['f1']:.3f}  AUC {row['auc']:.3f} | thr {row['threshold']:.2f} | {row['chosen']:10s} lr {row['lr_ft']}"
          f" | val_loss {row['val_loss_best']:.4f} | {row['train_min']:.1f} min")

## 2. Train the two Early variants
A result line appears after each run. Re-running continues where it left off.

In [ ]:
RES, PRED, EFF = Path("results_per_fold.csv"), Path("test_predictions_all.csv"), Path("efficiency.csv")
queue = CONFIG["VARIANTS"]
done = set(map(tuple, pd.read_csv(RES)[["model", "fold"]].values)) if RES.exists() else set()
env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="2")

t_start = time.time()
for name in queue:
    for k in CONFIG["FOLDS"]:
        if (name, k) in done:
            continue
        for attempt in (1, 2):                     # one automatic retry if a run fails
            r = subprocess.run([sys.executable, "train_one.py", name, str(k)], capture_output=True, text=True, env=env)
            result = [l for l in r.stdout.splitlines() if l.startswith("RESULT ")]
            if r.returncode == 0 and result:
                print(result[-1][7:] + f" | total {(time.time() - t_start) / 60:.0f} min", flush=True)
                break
            print(f"!! {name} fold {k}: attempt {attempt} failed (exit code {r.returncode}). Last error lines:")
            print("\n".join((r.stderr or r.stdout).splitlines()[-15:]), flush=True)
        else:
            raise RuntimeError(f"{name} fold {k} failed twice - see the error lines above")
print("All runs finished.")

# ---- merge with notebook 06 (its 25 runs are copied unchanged) ----
w06 = next(Path("/kaggle/input").rglob("Baseline_fold0.npz")).parent
OUT06 = w06.parent
assert "selected_attention" in json.load(open(OUT06 / "selection.json")), "Attach DFU 06 - LiteNet Ablation"
selected_06 = json.load(open(OUT06 / "selection.json"))["selected_attention"]
for fname in ["results_per_fold.csv", "test_predictions_all.csv", "efficiency.csv"]:
    old, new = pd.read_csv(OUT06 / fname), pd.read_csv(fname)
    pd.concat([old[~old.model.isin(set(new.model))], new], ignore_index=True).to_csv(fname.replace(".csv", "_all.csv"), index=False)
RES, PRED, EFF = Path("results_per_fold_all.csv"), Path("test_predictions_all_all.csv"), Path("efficiency_all.csv")
queue = ["Baseline", "ECA-Early", "CA-Early", "ECA-Late", "CA-Late", "ECA-Multi", "CA-Multi"]
r = pd.read_csv(RES)
assert sorted(r.model.unique()) == sorted(queue) and (r.groupby("model").fold.nunique() == 5).all(), "missing runs"
print("Merged: 7 variants x 5 folds. Selected in notebook 06:", selected_06)

## 3. Results for all 7 variants: mean ± standard deviation over the 5 folds

In [ ]:
res = pd.read_csv(RES); eff = pd.read_csv(EFF); pred_all = pd.read_csv(PRED)
metrics = ["accuracy", "sensitivity", "specificity", "precision", "f1", "auc"]
agg = res.groupby("model")[metrics].agg(["mean", "std"])
table = pd.DataFrame({m: [f"{100 * agg.loc[n, (m, 'mean')]:.1f} ± {100 * agg.loc[n, (m, 'std')]:.1f}" for n in queue] for m in metrics}, index=queue)
table["val_loss"] = res.groupby("model").val_loss_best.mean().loc[queue].round(4)
table = table.join(eff.set_index("model")[["params_M", "size_mb_fp32", "gflops", "latency_gpu_ms", "latency_cpu_ms"]].round(3))
table.to_csv("summary_table_all7.csv")
table

## 4. Ablation table: what does each placement add?
Changes are relative to the Baseline. McNemar's test compares per-image correctness on all 713 images.

In [ ]:
rows = []
base = pred_all[pred_all.model == "Baseline"].set_index(["file", "class"])
b_f1, b_params = agg.loc["Baseline", ("f1", "mean")], eff.set_index("model").loc["Baseline", "params_M"]
for n in queue:
    d = pred_all[pred_all.model == n].set_index(["file", "class"]).loc[base.index]
    fn = int(((d.label == 1) & (d.pred == 0)).sum()); fp = int(((d.label == 0) & (d.pred == 1)).sum())
    ok_b, ok_v = base.pred == base.label, d.pred == d.label
    b, c = int((ok_b & ~ok_v).sum()), int((~ok_b & ok_v).sum())   # baseline right & variant wrong / vice versa
    rows.append({"variant": n, "F1 change (points)": round(100 * (agg.loc[n, ("f1", "mean")] - b_f1), 2),
                 "missed ulcers": fn, "false alarms": fp, "AUC (pooled)": round(roc_auc_score(d.label, d.prob_ulcer), 4),
                 "extra params": int(round((eff.set_index("model").loc[n, "params_M"] - b_params) * 1e6)),
                 "baseline right, variant wrong": b, "variant right, baseline wrong": c,
                 "McNemar p": 1.0 if n == "Baseline" or b + c == 0 else round(binomtest(b, b + c, 0.5).pvalue, 4)})
ablation = pd.DataFrame(rows).set_index("variant")
ablation.to_csv("ablation_table_all7.csv")
ablation

## 5. Selection and sensitivity check

In [ ]:
def select(r):
    # The rule fixed in advance: lowest mean validation loss among attention variants
    v = r[r.model != "Baseline"].groupby("model").val_loss_best.mean()
    return v.idxmin()

selected, selected14 = select(res), select(res[res.fold != 0])
print(f"Pre-registered DFU-LiteNet variant (notebook 06): {selected_06}")
print(f"Same rule applied to all 6 attention variants: {selected} -> "
      + ("SAME" if selected == selected_06 else "DIFFERENT: report it; notebook 06's choice stays the pre-registered one"))
vl = res.groupby("model").val_loss_best.agg(["mean", "std"]).loc[queue]
p_sel = float(ablation.loc[selected, "McNemar p"])
print(vl.round(4).to_string(), "\n")
print(f"Lowest mean validation loss among all 6 attention variants: {selected}")
print(f"Without fold 0: {selected14} -> " + ("SAME" if selected14 == selected else "DIFFERENT: discuss this in the thesis"))
print(f"Compared with the Baseline: F1 change {ablation.loc[selected, 'F1 change (points)']:+.2f} points, McNemar p = {p_sel:.4f} -> "
      + ("statistically significant" if p_sel < 0.05 else "NOT statistically significant"))

lr_table = res.assign(choice=res.apply(lambda r: r.chosen if r.chosen != "fine-tuned" else f"lr {r.lr_ft:g}", axis=1))
print("\nChosen fine-tuning learning rate per fold:")
print(lr_table.pivot(index="model", columns="fold", values="choice").loc[queue].to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
means, stds = 100 * agg.loc[queue, ("f1", "mean")], 100 * agg.loc[queue, ("f1", "std")]
colors_ = ["#888888" if n == "Baseline" else "#b22222" if n == selected else "#1f3b63" for n in queue]
ax[0].bar(queue, means, yerr=stds, color=colors_, capsize=4); ax[0].set_ylim(max(90, (means - stds).min() - 1), 100.5)
ax[0].set_ylabel("F1 (%), mean ± std over 5 folds"); ax[0].set_title("F1 per variant (grey = baseline, red = selected)")
ax[1].bar(queue, vl["mean"], yerr=vl["std"], color=colors_, capsize=4)
ax[1].set_ylabel("Best validation loss"); ax[1].set_title("Validation loss used for selection")
plt.tight_layout(); plt.savefig("ablation_chart_all7.png", dpi=130); plt.show()

json.dump({"selected_attention_nb06": selected_06, "rule_choice_all6": selected, "rule_choice_all6_without_fold0": selected14,
           "mcnemar_p_vs_baseline_rule_choice": p_sel, "rule": "lowest mean validation loss among attention variants"},
          open("selection_06b.json", "w"), indent=2)
print(open("selection_06b.json").read())

## What to check
- **The result lines:** all 10 runs should finish.
- **Section 3-4:** where Early sits compared with Baseline, Late and Multi. McNemar p below 0.05 means a real difference.
- **Section 5:** whether the same selection rule would pick a different variant among all six. Notebook 06's choice remains the pre-registered DFU-LiteNet either way.

**Outputs:** `results_per_fold.csv` (new runs), `*_all.csv` (all 7 variants), `ablation_table_all7.csv`, `summary_table_all7.csv`, `selection_06b.json`, and `weights/` for the two Early variants.